# 03 — MILP Formulation & Solution

Formulates and solves the Dynamic Pricing & Inventory Allocation problem as a Mixed-Integer Linear Program using OR-Tools CP-SAT.

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from data_prep import load_processed
from demand_model import DemandModel
from milp_solver import solve_pricing_milp, create_milp_from_data

plt.rcParams['figure.figsize'] = (12, 6)

## 1. Load Data and Model

In [ ]:
weekly_df, elasticity_df = load_processed()
demand_model = DemandModel.load(model_type='xgboost')

print(f"Data: {weekly_df.shape}")
print(f"SKUs: {weekly_df['product_id'].nunique()}")
print(f"Locations: {weekly_df['location'].nunique()}")
print(f"Weeks: {weekly_df['week'].nunique()}")

## 2. Define Business Constraints

In [ ]:
location_list = weekly_df['location'].unique().tolist()
capacities = {loc: 800 for loc in location_list}
print(f"Location capacities: {capacities}")

MAX_PRICE_CHANGE = 15.0
MIN_STOCK = 5
N_PRICE_LEVELS = 8
print(f"Max price change: ${MAX_PRICE_CHANGE}")
print(f"Min stock: {MIN_STOCK} units")
print(f"Price levels per SKU: {N_PRICE_LEVELS}")

## 3. Formulate and Solve MILP

In [ ]:
milp_result = solve_pricing_milp(
    weekly_df, demand_model,
    capacities=capacities,
    n_price_levels=N_PRICE_LEVELS,
    max_price_change=MAX_PRICE_CHANGE,
    min_stock=MIN_STOCK,
    timeout_seconds=120
)

## 4. Analyze Solution

In [ ]:
if milp_result['assignments']:
    assignments_df = pd.DataFrame([
        {'sku': k[0], 'location': k[1], 'week': k[2],
         'price': v['price'], 'allocation': v['allocation'],
         'revenue': v['revenue']}
        for k, v in milp_result['assignments'].items()
    ])
    
    print(f"Total assignments: {len(assignments_df)}")
    print(f"Total revenue: ${assignments_df['revenue'].sum():,.2f}")
    print(f"Total allocation: {assignments_df['allocation'].sum():,} units")
    print(f"\nAverage price per SKU:")
    print(assignments_df.groupby('sku')['price'].mean().round(2))
    
    assignments_df.head(10)

## 5. Price Distribution Across Locations

In [ ]:
if milp_result['assignments']:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    assignments_df.groupby('location')['price'].mean().plot(kind='bar', ax=axes[0], color='steelblue')
    axes[0].set_title('Average Optimized Price by Location')
    axes[0].set_ylabel('Price ($)')
    
    assignments_df.groupby('location')['revenue'].sum().plot(kind='bar', ax=axes[1], color='coral')
    axes[1].set_title('Total Revenue by Location')
    axes[1].set_ylabel('Revenue ($)')
    
    plt.tight_layout()
    plt.show()